In [8]:
import pandas as pd

data_path = "../data/processed/filtered_complaints.csv"
df = pd.read_csv(data_path)

df.shape


(570107, 21)

In [9]:
df["product_category"].value_counts(normalize=True)


product_category
Credit card        0.332103
Savings account    0.272237
Personal loan      0.225188
Money transfer     0.170473
Name: proportion, dtype: float64

In [10]:
SAMPLE_SIZE = 12_000

sampled_df = (
    df
    .groupby("product_category", group_keys=False)
    .apply(lambda x: x.sample(
        n=int(len(x) / len(df) * SAMPLE_SIZE),
        random_state=42
    ))
)

sampled_df.shape


C:\Users\Admin\AppData\Local\Temp\ipykernel_31580\3765562726.py:6: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(


(11998, 21)

In [11]:
# Check category proportions
sampled_df["product_category"].value_counts(normalize=True)


product_category
Credit card        0.332139
Savings account    0.272212
Personal loan      0.225204
Money transfer     0.170445
Name: proportion, dtype: float64

In [12]:
output_path = "../data/processed/sample_complaints_12k.csv"
sampled_df.to_csv(output_path, index=False)

print(f"Saved stratified sample to {output_path}")


Saved stratified sample to ../data/processed/sample_complaints_12k.csv


In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [14]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=["\n\n", "\n", " ", ""]
)


In [15]:
def chunk_complaint(row):
    chunks = text_splitter.split_text(row["clean_narrative"])
    return [
        {
            "text": chunk,
            "complaint_id": row["Complaint ID"],
            "product_category": row["product_category"],
            "chunk_index": idx,
            "total_chunks": len(chunks)
        }
        for idx, chunk in enumerate(chunks)
    ]


In [16]:
all_chunks = []

for _, row in sampled_df.iterrows():
    all_chunks.extend(chunk_complaint(row))

len(all_chunks)


35509

In [17]:
chunks_df = pd.DataFrame(all_chunks)
chunks_df.head()


,text,complaint_id,product_category,chunk_index,total_chunks
0,my soon to be ex wife took out several credit ...,7471816,Credit card,0,1
1,after informing bank of america of my identity...,3952712,Credit card,0,1
2,i am deeply troubled by the inclusion of this ...,9669839,Credit card,0,1
3,on xx xx xxxx i reached out to discover bank t...,7420081,Credit card,0,23
4,which put me in a positive balance since i had...,7420081,Credit card,1,23


In [19]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

C:\Users\Admin\Desktop\kifiya-tasks\rag-complaint-chatbot\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Admin\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [20]:
texts = chunks_df["text"].tolist()

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    batch_size=64
)

len(embeddings), embeddings[0].shape


Batches:   0%|          | 0/555 [00:00<?, ?it/s]

(35509, (384,))

In [21]:
metadatas = chunks_df[[
    "complaint_id",
    "product_category",
    "chunk_index",
    "total_chunks"
]].to_dict(orient="records")


In [22]:
import chromadb
from chromadb.config import Settings

client = chromadb.Client(
    Settings(
        persist_directory="../vector_store/chroma",
        anonymized_telemetry=False
    )
)

collection = client.get_or_create_collection(
    name="complaint_chunks",
    metadata={"hnsw:space": "cosine"}
)


In [23]:
collection.add(
    documents=texts,
    embeddings=embeddings.tolist(),
    metadatas=metadatas,
    ids=[f"chunk_{i}" for i in range(len(texts))]
)

collection.count()


InternalError: ValueError: Batch size of 35509 is greater than max batch size of 5461